# 직접 수집 1 · 데이터 수집 ① — 시드 & 네트워크 확장

위키피디아를 직접 크롤링해 유망아이템을 발굴하는 과정을 **단계별로** 실행합니다.
전체 흐름: **수집①(네트워크) → 정제(필터) → 수집②(지표) → 결과(PageRank·통계) → 보고서**

## 이 노트북 (수집 ①)
1. **시드 실제 제목 확인** — 입력어를 위키의 정식 문서명으로 정규화
2. **네트워크 확장** — 시드 문서의 **"See also"** 링크를 따라 연관 문서를 모음

> 이후 노트북(02~05)은 이 결과 파일을 이어받습니다. **순서대로 실행**하세요.

In [ ]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # pipeline.py 있는 프로젝트 루트 탐색
    if os.path.isfile("pipeline.py"):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 1                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

## 1) 시드 실제 제목 확인
입력한 이름이 리다이렉트/표기 차이가 있을 수 있어, 위키의 **정식 문서명**으로 맞춥니다.

In [ ]:
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("입력:", SEED, "→ 실제 제목:", SEED_TRUE)
df_true.head()

## 2) 네트워크 확장의 원리 — "See also"
확장은 문서 위키텍스트의 **`== See also ==`** 섹션 링크를 따라갑니다.
(일반 `[[링크]]` + `{{Annotated link|...}}` 템플릿 모두 인식하도록 파서를 보강했습니다.)

먼저 시드 문서의 See also 링크를 직접 뽑아봅니다.

In [ ]:
sess = wc._create_session()
r = sess.get("https://en.wikipedia.org/w/api.php", params={
    "action": "query", "format": "json", "prop": "revisions",
    "rvprop": "content", "rvslots": "main", "titles": SEED_TRUE, "formatversion": "2"
}, verify=False, timeout=25)
wikitext = r.json()["query"]["pages"][0]["revisions"][0]["slots"]["main"]["content"]

see_also = wc._parse_see_also(wikitext)
print(f"'{SEED_TRUE}'의 See also 링크: {len(see_also)}개")
see_also

## 3) 전체 네트워크 확장 실행 (1차수)
`expand_network`가 시드의 See also 문서들을 크롤링해 **엣지(시드 → 연관문서)** 표를 만듭니다.

In [ ]:
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
edf = pd.read_excel(EXPAND)
nodes = set(edf["From_title"].dropna()) | set(edf["To_seealso"].dropna())
print(f"확장 엣지 {len(edf)}개 · 노드 {len(nodes)}개  →  {EXPAND}")
edf.head(15)

**정리**
- 시드 → See also 연관 문서로 **네트워크(엣지)** 를 확보했습니다.
- 다음 노트북(**02 정제**)에서 목록성 문서 등 노이즈를 걸러냅니다.